# LitSpace Evaluation Analysis

This notebook goes through the committed evaluation results for LitSpace. It loads the saved metrics, regenerates the plots used in the README, and explains what the numbers say about the current version of the system.

The point is not just to show the best-looking results. It is also to be clear about where LitSpace is working well, where it is only partly working, and where the same weak spots keep showing up.


## Notebook Roadmap

The notebook follows a simple order:

1. Set up paths and load the saved results.
2. Look at what is actually in the benchmark.
3. Compare LitSpace to the two baselines overall.
4. Separate retrieval performance from answer quality.
5. Look at pairwise wins, category-level results, and failure types.
6. End with the quick demo path.

## Useful Commands

Use these commands from the repository root.

```bash
# Open the notebook
jupyter notebook evaluation/notebooks/evaluation_analysis.ipynb

# Quick demo: create a demo project and run 3 sample questions
backend/.litenv/bin/python demo/quick_demo.py

# Static fallback from committed outputs
backend/.litenv/bin/python demo/quick_demo.py --static

# Full evaluation pipeline
backend/.litenv/bin/python evaluation/scripts/setup_project.py
backend/.litenv/bin/python evaluation/scripts/run_systems.py
backend/.litenv/bin/python evaluation/scripts/judge_answers.py
backend/.litenv/bin/python evaluation/scripts/pairwise_judge.py
backend/.litenv/bin/python evaluation/scripts/summarize_results.py
```


## 1. Setup and Paths

This part imports the standard-library modules used in the notebook and sets the main paths. The path logic is written so the notebook still works whether it is opened from the repo root or from inside `evaluation/notebooks/`.

Main files used here:

- `evaluation/results/metrics_summary.json` for the aggregated metrics
- `evaluation/results/error_analysis.csv` for row-level failure labels
- `evaluation/datasets/questions.csv` for benchmark composition
- `evaluation/results/plots/` for the saved figures this notebook regenerates


In [1]:
from __future__ import annotations

import csv
import json
from collections import Counter
from html import escape
from pathlib import Path

# Resolve the repo root once so the rest of the paths are stable.
# This works from the repo root and from evaluation/notebooks.
ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
EVAL_DIR = ROOT / 'evaluation'
RESULTS_DIR = EVAL_DIR / 'results'
PLOTS_DIR = RESULTS_DIR / 'plots'
SUMMARY_JSON = RESULTS_DIR / 'metrics_summary.json'
ERROR_CSV = RESULTS_DIR / 'error_analysis.csv'
QUESTIONS_CSV = EVAL_DIR / 'datasets' / 'questions.csv'

print('Root:', ROOT)
print('Summary file:', SUMMARY_JSON)
print('Error file:', ERROR_CSV)
print('Questions file:', QUESTIONS_CSV)
print('Plots dir:', PLOTS_DIR)


Root: /Users/enasbatarfi/DS593-LLM/litspace
Summary file: /Users/enasbatarfi/DS593-LLM/litspace/evaluation/results/metrics_summary.json
Error file: /Users/enasbatarfi/DS593-LLM/litspace/evaluation/results/error_analysis.csv
Questions file: /Users/enasbatarfi/DS593-LLM/litspace/evaluation/datasets/questions.csv
Plots dir: /Users/enasbatarfi/DS593-LLM/litspace/evaluation/results/plots


## 2. Load the Main Results

The summary file already contains the main outputs from the evaluation pipeline. It includes:

- overall averages for each system
- pairwise comparison rates
- category-level breakdowns

Loading them once here keeps the rest of the notebook focused on the results instead of file handling.


In [2]:
# Load the saved metrics used throughout the notebook.
metrics = json.loads(SUMMARY_JSON.read_text(encoding='utf-8'))
overall = metrics['overall']
pairwise = metrics['pairwise']
by_category = metrics['by_category']

# Make sure the plot directory exists before saving refreshed SVGs.
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print('Systems found:', ', '.join(overall.keys()))
print('Pairwise sets:', ', '.join(pairwise.keys()))
print('Categories:', ', '.join(by_category.keys()))


Systems found: litspace_rag, zero_shot, summary_few_shot
Pairwise sets: litspace_vs_zero_shot, litspace_vs_summary_few_shot
Categories: ambiguity, evidence_lookup, followup, multi_paper_synthesis, refusal, single_paper_factual, single_paper_summary


## 3. Benchmark Overview

Before looking at the model scores, it helps to look at the benchmark itself. The evaluation set has 45 questions spread across seven categories: factual questions, summaries, multi-paper synthesis, evidence lookup, follow-up behavior, ambiguity handling, and refusal behavior.

That matters because LitSpace is trying to do more than one thing. A system can do well on direct factual questions and still struggle when it has to combine papers or decide whether it should clarify first.


In [3]:
# Count examples in each benchmark category.
# This helps make sense of the category results later.
category_counts = Counter()
with QUESTIONS_CSV.open('r', encoding='utf-8', newline='') as handle:
    reader = csv.DictReader(handle)
    for row in reader:
        category_counts[row['category']] += 1

category_order = [
    'single_paper_factual',
    'single_paper_summary',
    'multi_paper_synthesis',
    'evidence_lookup',
    'followup',
    'ambiguity',
    'refusal',
]

benchmark_rows = [(category, category_counts[category]) for category in category_order]
benchmark_rows


[('single_paper_factual', 7),
 ('single_paper_summary', 7),
 ('multi_paper_synthesis', 8),
 ('evidence_lookup', 7),
 ('followup', 7),
 ('ambiguity', 5),
 ('refusal', 4)]

| Category | Questions |
| --- | ---: |
| Single-paper factual | 7 |
| Single-paper summary | 7 |
| Multi-paper synthesis | 8 |
| Evidence lookup | 7 |
| Follow-up | 7 |
| Ambiguity | 5 |
| Refusal | 4 |
| **Total** | **45** |

The benchmark is fairly balanced across the main behaviors the project is trying to support, although ambiguity and refusal have fewer examples than the other categories. So those two categories are still useful, but they should be read a little more carefully than the larger ones.


## 4. Overall System Comparison

This is the main baseline check: how does LitSpace compare to the two non-retrieval baselines on overall answer quality?

A few things matter when reading this table:

- correctness, completeness, relevance, helpfulness, and faithfulness are judge-based scores
- correctness is on a 0 to 2 scale
- the point is not only whether LitSpace wins, but also where the stronger baseline stays competitive


In [4]:
# Keep shared labels and colors in one place.
SYSTEM_LABELS = {
    'litspace_rag': 'LitSpace RAG',
    'zero_shot': 'Zero-shot',
    'summary_few_shot': 'Summary few-shot',
}

SYSTEM_COLORS = {
    'litspace_rag': '#0f766e',
    'zero_shot': '#c2410c',
    'summary_few_shot': '#1d4ed8',
}


def svg_document(width: int, height: int, body: str) -> str:
    """Wrap SVG body content with the shared notebook chart style."""
    return (
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" '
        f'viewBox="0 0 {width} {height}" role="img">'
        '<style>'
        '.title{font:700 24px Arial,sans-serif;fill:#0f172a;}'
        '.subtitle{font:400 13px Arial,sans-serif;fill:#475569;}'
        '.label{font:600 13px Arial,sans-serif;fill:#0f172a;}'
        '.tick{font:400 12px Arial,sans-serif;fill:#64748b;}'
        '.value{font:700 12px Arial,sans-serif;fill:#0f172a;}'
        '.foot{font:400 11px Arial,sans-serif;fill:#64748b;}'
        '</style>'
        f'{body}</svg>'
    )


def horizontal_bar_chart(*, title: str, subtitle: str, items: list[dict], output_path: Path, max_value: float, value_fmt: str, footnote: str) -> None:
    """Save a horizontal bar chart as SVG."""
    width = 960
    top = 90
    left = 280
    right = 80
    row_h = 52
    chart_h = row_h * len(items)
    height = top + chart_h + 70
    chart_w = width - left - right

    # Add reference lines so the bar lengths are easier to compare.
    grid_values = [0, max_value * 0.25, max_value * 0.5, max_value * 0.75, max_value]
    body = [
        f'<rect x="0" y="0" width="{width}" height="{height}" fill="#f8fafc" />',
        f'<text x="36" y="42" class="title">{escape(title)}</text>',
        f'<text x="36" y="64" class="subtitle">{escape(subtitle)}</text>',
    ]

    for grid_value in grid_values:
        x = left + (grid_value / max_value) * chart_w if max_value else left
        label = value_fmt.format(grid_value)
        body.append(f'<line x1="{x:.2f}" y1="{top - 10}" x2="{x:.2f}" y2="{top + chart_h}" stroke="#e2e8f0" stroke-width="1" />')
        body.append(f'<text x="{x:.2f}" y="{top + chart_h + 20}" class="tick" text-anchor="middle">{escape(label)}</text>')

    for index, item in enumerate(items):
        y = top + index * row_h
        bar_y = y + 8
        bar_h = 24
        bar_w = (item['value'] / max_value) * chart_w if max_value else 0
        value_text = value_fmt.format(item['value'])
        body.append(f'<text x="{left - 16}" y="{bar_y + 16}" class="label" text-anchor="end">{escape(item["label"])}</text>')
        body.append(f'<rect x="{left}" y="{bar_y}" width="{chart_w}" height="{bar_h}" rx="8" fill="#e2e8f0" />')
        body.append(f'<rect x="{left}" y="{bar_y}" width="{bar_w:.2f}" height="{bar_h}" rx="8" fill="{item["color"]}" />')
        body.append(f'<text x="{left + bar_w + 10:.2f}" y="{bar_y + 16}" class="value">{escape(value_text)}</text>')

    body.append(f'<text x="36" y="{height - 22}" class="foot">{escape(footnote)}</text>')
    output_path.write_text(svg_document(width, height, ''.join(body)), encoding='utf-8')


In [5]:
# Save the overall comparison plot used in the README.
overall_plot = PLOTS_DIR / 'overall_correctness.svg'

horizontal_bar_chart(
    title='Overall Correctness Mean',
    subtitle='Higher is better. Values come from evaluation/results/metrics_summary.json.',
    items=[
        {'label': SYSTEM_LABELS['litspace_rag'], 'value': overall['litspace_rag']['correctness_mean'], 'color': SYSTEM_COLORS['litspace_rag']},
        {'label': SYSTEM_LABELS['summary_few_shot'], 'value': overall['summary_few_shot']['correctness_mean'], 'color': SYSTEM_COLORS['summary_few_shot']},
        {'label': SYSTEM_LABELS['zero_shot'], 'value': overall['zero_shot']['correctness_mean'], 'color': SYSTEM_COLORS['zero_shot']},
    ],
    output_path=overall_plot,
    max_value=2.0,
    value_fmt='{:.2f}',
    footnote='Judge score range: 0 to 2.',
)

print('Saved plot:', overall_plot)


Saved plot: /Users/enasbatarfi/DS593-LLM/litspace/evaluation/results/plots/overall_correctness.svg


| System | Correctness | Completeness | Relevance | Helpfulness | Faithfulness | Avg latency (s) |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| LitSpace RAG | 1.44 | 1.22 | 1.73 | 1.40 | 1.49 | 2.86 |
| Summary few-shot | 1.33 | 1.09 | 1.80 | 1.29 | 1.71 | 1.92 |
| Zero-shot | 0.33 | 0.22 | 1.22 | 0.31 | N/A | 2.05 |

![Overall correctness](../results/plots/overall_correctness.svg)

This table shows LitSpace clearly beating zero-shot. The gap is large on correctness, completeness, and helpfulness, so retrieval is making a real difference here.

The comparison with summary few-shot is closer. LitSpace is a bit better on correctness and completeness, but summary few-shot is still stronger on relevance and faithfulness. So the stronger baseline is still competitive here. A fair read is that LitSpace is better overall, but not by a huge margin on every metric.


## 5. Retrieval Performance

The overall scores show answer quality. This section looks at retrieval directly so it is easier to see where LitSpace is strong and where it is still losing information.

The key split here is paper-level retrieval versus section-level retrieval. Finding the right paper is necessary, but that is not the same as finding the exact section needed to support the answer well.


In [6]:
# Save the retrieval metrics plot for LitSpace.
litspace = overall['litspace_rag']
retrieval_plot = PLOTS_DIR / 'litspace_retrieval_metrics.svg'

horizontal_bar_chart(
    title='LitSpace Retrieval Metrics',
    subtitle='Paper-level retrieval is strong; exact supporting-section retrieval is weaker.',
    items=[
        {'label': 'Paper hit@5', 'value': litspace['paper_hit_at_5_mean'], 'color': '#0f766e'},
        {'label': 'Paper recall@5', 'value': litspace['paper_recall_at_5_mean'], 'color': '#0f766e'},
        {'label': 'Section hit@5', 'value': litspace['section_hit_at_5_mean'], 'color': '#1d4ed8'},
        {'label': 'Section recall@5', 'value': litspace['section_recall_at_5_mean'], 'color': '#1d4ed8'},
    ],
    output_path=retrieval_plot,
    max_value=1.0,
    value_fmt='{:.2f}',
    footnote='All values are proportions between 0 and 1.',
)

print('Saved plot:', retrieval_plot)


Saved plot: /Users/enasbatarfi/DS593-LLM/litspace/evaluation/results/plots/litspace_retrieval_metrics.svg


| Metric | LitSpace |
| --- | ---: |
| Paper hit@5 | 0.97 |
| Paper recall@5 | 0.86 |
| Section hit@5 | 0.64 |
| Section recall@5 | 0.31 |
| Paper MRR@5 | 0.95 |

![Retrieval metrics](../results/plots/litspace_retrieval_metrics.svg)

Paper retrieval is strong. LitSpace almost always brings in a relevant paper, and the high MRR means the correct paper is usually near the top of the ranking.

Section retrieval is weaker, especially section recall at 0.31. That means the system often finds the right paper but still misses some of the exact supporting sections it needs. This matters because retrieval is no longer the main bottleneck at the paper level. The weaker part is getting the right evidence inside the paper and then using it cleanly in the answer.


## 6. Pairwise Comparison Against Baselines

Average scores are useful, but pairwise comparisons answer a different question: if two answers are put side by side, how often does LitSpace win?

This helps show whether the overall advantage is broad across the benchmark or driven by a smaller number of big wins.


In [7]:
# Define the stacked-share chart here because this is the only section that uses it.
def stacked_share_chart(*, title: str, subtitle: str, rows: list[dict], output_path: Path, footnote: str) -> None:
    """Save a stacked proportion chart as SVG."""
    width = 960
    top = 92
    left = 250
    right = 80
    row_h = 64
    chart_h = row_h * len(rows)
    height = top + chart_h + 94
    chart_w = width - left - right
    legend_y = height - 54
    legend_items = [
        ('LitSpace win', '#0f766e'),
        ('Baseline win', '#c2410c'),
        ('Tie', '#94a3b8'),
    ]
    body = [
        f'<rect x="0" y="0" width="{width}" height="{height}" fill="#f8fafc" />',
        f'<text x="36" y="42" class="title">{escape(title)}</text>',
        f'<text x="36" y="64" class="subtitle">{escape(subtitle)}</text>',
    ]

    # Add percentage guides so each row reads like a full 0-100% bar.
    for pct in (0, 0.25, 0.5, 0.75, 1.0):
        x = left + pct * chart_w
        body.append(f'<line x1="{x:.2f}" y1="{top - 12}" x2="{x:.2f}" y2="{top + chart_h}" stroke="#e2e8f0" stroke-width="1" />')
        body.append(f'<text x="{x:.2f}" y="{top + chart_h + 22}" class="tick" text-anchor="middle">{int(pct * 100)}%</text>')

    for index, row in enumerate(rows):
        y = top + index * row_h
        bar_y = y + 10
        bar_h = 26
        body.append(f'<text x="{left - 16}" y="{bar_y + 17}" class="label" text-anchor="end">{escape(row["label"])}</text>')
        current_x = left
        for segment in row['segments']:
            segment_w = chart_w * segment['value']
            body.append(f'<rect x="{current_x:.2f}" y="{bar_y}" width="{segment_w:.2f}" height="{bar_h}" fill="{segment["color"]}" />')
            if segment_w >= 68:
                body.append(f'<text x="{current_x + (segment_w / 2):.2f}" y="{bar_y + 17}" class="tick" text-anchor="middle" fill="#ffffff">{segment["label"]}</text>')
            current_x += segment_w

    legend_x = 36
    for label, color in legend_items:
        body.append(f'<rect x="{legend_x}" y="{legend_y}" width="16" height="16" rx="3" fill="{color}" />')
        body.append(f'<text x="{legend_x + 24}" y="{legend_y + 13}" class="tick">{escape(label)}</text>')
        legend_x += 170

    body.append(f'<text x="36" y="{height - 16}" class="foot">{escape(footnote)}</text>')
    output_path.write_text(svg_document(width, height, ''.join(body)), encoding='utf-8')


In [8]:
# Save the pairwise win-rate plot.
pairwise_plot = PLOTS_DIR / 'pairwise_win_rates.svg'

stacked_share_chart(
    title='Pairwise Comparison Against Baselines',
    subtitle='Each row shows LitSpace win rate, baseline win rate, and tie rate.',
    rows=[
        {
            'label': 'LitSpace vs zero-shot',
            'segments': [
                {'label': '75.6%', 'value': pairwise['litspace_vs_zero_shot']['a_win_rate'], 'color': '#0f766e'},
                {'label': '13.3%', 'value': pairwise['litspace_vs_zero_shot']['b_win_rate'], 'color': '#c2410c'},
                {'label': '11.1%', 'value': pairwise['litspace_vs_zero_shot']['tie_rate'], 'color': '#94a3b8'},
            ],
        },
        {
            'label': 'LitSpace vs summary few-shot',
            'segments': [
                {'label': '48.9%', 'value': pairwise['litspace_vs_summary_few_shot']['a_win_rate'], 'color': '#0f766e'},
                {'label': '33.3%', 'value': pairwise['litspace_vs_summary_few_shot']['b_win_rate'], 'color': '#c2410c'},
                {'label': '17.8%', 'value': pairwise['litspace_vs_summary_few_shot']['tie_rate'], 'color': '#94a3b8'},
            ],
        },
    ],
    output_path=pairwise_plot,
    footnote='Pairwise values are taken directly from metrics_summary.json.',
)

print('Saved plot:', pairwise_plot)


Saved plot: /Users/enasbatarfi/DS593-LLM/litspace/evaluation/results/plots/pairwise_win_rates.svg


| Comparison | LitSpace win rate | Baseline win rate | Tie rate |
| --- | ---: | ---: | ---: |
| LitSpace vs zero-shot | 0.756 | 0.133 | 0.111 |
| LitSpace vs summary few-shot | 0.489 | 0.333 | 0.178 |

![Pairwise comparison](../results/plots/pairwise_win_rates.svg)

This plot says the same basic thing as the overall table, but in a more direct way. LitSpace beats zero-shot on most examples, which supports the claim that retrieval is helping in a consistent way.

The summary few-shot comparison is much closer. LitSpace still wins more often than it loses, but the gap is smaller and the tie rate is not tiny. So the result is positive for LitSpace, but it is still a mixed comparison rather than a blowout.


## 7. Category-Level Breakdown

The overall averages hide a lot. This section breaks correctness down by question type so it is easier to see where the system is strong and where it is still weaker.

That matters because LitSpace is doing several different tasks, and strong performance in one category does not automatically carry over to another.


In [9]:
# Expand category names so the chart labels read cleanly.
CATEGORY_LABELS = {
    'single_paper_factual': 'Single-paper factual',
    'single_paper_summary': 'Single-paper summary',
    'multi_paper_synthesis': 'Multi-paper synthesis',
    'evidence_lookup': 'Evidence lookup',
    'followup': 'Follow-up',
    'ambiguity': 'Ambiguity',
    'refusal': 'Refusal',
}

category_plot = PLOTS_DIR / 'litspace_category_correctness.svg'

# Use a second color for the weaker categories so they stand out.
horizontal_bar_chart(
    title='LitSpace Correctness by Question Category',
    subtitle='This chart surfaces where LitSpace is strongest and where it struggles.',
    items=[
        {
            'label': CATEGORY_LABELS[category],
            'value': by_category[category]['litspace_rag']['correctness_mean'],
            'color': '#0f766e' if category not in {'ambiguity', 'multi_paper_synthesis'} else '#b45309',
        }
        for category in category_order
    ],
    output_path=category_plot,
    max_value=2.0,
    value_fmt='{:.2f}',
    footnote='Ambiguity and multi-paper synthesis are the clearest weak spots in the current benchmark.',
)

print('Saved plot:', category_plot)


Saved plot: /Users/enasbatarfi/DS593-LLM/litspace/evaluation/results/plots/litspace_category_correctness.svg


| Category | LitSpace correctness |
| --- | ---: |
| Single-paper factual | 1.57 |
| Single-paper summary | 1.71 |
| Multi-paper synthesis | 0.88 |
| Evidence lookup | 1.71 |
| Follow-up | 2.00 |
| Ambiguity | 0.60 |
| Refusal | 1.50 |

![Category breakdown](../results/plots/litspace_category_correctness.svg)

The strongest categories here are follow-up, evidence lookup, and the single-paper tasks. Those are all settings where the system can stay closer to one paper or one conversation thread, and the scores look solid there.

The weaker categories are ambiguity and multi-paper synthesis. Both need more than basic retrieval. Ambiguity needs good judgment about when to clarify, and multi-paper synthesis needs cleaner combining of evidence across papers. These are still some of the weaker parts of the system.


## 8. Failure Analysis

The last section moves from averages to row-level failure labels. The summary metrics show where performance drops, but the failure labels help show what the mistakes actually look like.

That makes this section useful for checking whether the weak categories match the actual error patterns in the saved results.


In [10]:
# Count LitSpace failures from the row-level error file.
# Skip rows labeled "good" so the chart stays focused on actual failures.
failure_counts = Counter()
with ERROR_CSV.open('r', encoding='utf-8', newline='') as handle:
    reader = csv.DictReader(handle)
    for row in reader:
        if row['system_name'] != 'litspace_rag':
            continue
        failure_type = row['failure_type'].strip()
        if failure_type and failure_type != 'good':
            failure_counts[failure_type] += 1

# Sort failures from most common to least common before plotting.
failure_items = [
    {'label': failure_type.replace('_', ' '), 'value': count, 'color': '#c2410c'}
    for failure_type, count in sorted(failure_counts.items(), key=lambda item: (-item[1], item[0]))
]

failure_plot = PLOTS_DIR / 'litspace_failure_types.svg'

horizontal_bar_chart(
    title='LitSpace Failure Types',
    subtitle='Counts are aggregated from evaluation/results/error_analysis.csv for LitSpace rows only.',
    items=failure_items,
    output_path=failure_plot,
    max_value=max(item['value'] for item in failure_items) if failure_items else 1,
    value_fmt='{:.0f}',
    footnote='The largest buckets are unsupported-claim cases, missing-key-point cases, and clarification-related failures.',
)

print('Saved plot:', failure_plot)
failure_counts


Saved plot: /Users/enasbatarfi/DS593-LLM/litspace/evaluation/results/plots/litspace_failure_types.svg


Counter({'unsupported_claim': 14,
         'missing_key_point': 11,
         'over_clarification': 4,
         'should_have_clarified': 3,
         'wrong_paper': 2,
         'over_refusal': 1,
         'should_have_refused': 1})

| Failure type | Count |
| --- | ---: |
| Unsupported claim | 14 |
| Missing key point | 11 |
| Over clarification | 4 |
| Should have clarified | 3 |
| Wrong paper | 2 |
| Over refusal | 1 |
| Should have refused | 1 |

![Failure types](../results/plots/litspace_failure_types.svg)

The two biggest failure types are unsupported claims and missing key points. That suggests a lot of the mistakes are partial-answer mistakes, not total breakdowns. The system is often in the right area, but the final answer is either not grounded enough or not complete enough.

The clarification-related failures line up with the weak ambiguity score from the category chart. LitSpace sometimes asks for clarification when it does not need to, and sometimes skips clarification when it probably should. So ambiguity is still a clear place where the system needs more work.


## 9. Quick Demo Notes

`demo/quick_demo.py` is the main demo entry point for the repo.

- by default, it starts the local backend if it is not already running
- it creates a fresh project from `demo/corpus/`
- it uploads the 7 demo PDFs, parses them, chunks them, and indexes them
- it creates a chat and asks 3 sample questions through the real backend
- the backend handles provider fallback automatically if no GPT key is available
- `--static` prints committed answers from `evaluation/outputs/litspace_outputs.jsonl`

The main path is intentionally simple: run one command and get a real demo project plus three example questions.
